# Vosk voice commands (EN / ES / RU)

Run the cells in order:

1. **Setup**: imports, commands, language names, model paths, starting values for the tuning sliders.
2. **Load models**: also checks that every command and language-name word exists in each model's vocabulary.
3. **Devices**: lists inputs and defines the helpers.
4. **Scan**: records briefly from every input while you talk, and shows which ones actually hear you.
5. **Mic tester**: pick a device, duration and gain, then Record. "Use these settings" passes the device and gain to the listener.
6. **Listener**: defines the background listener (does not block the notebook).
7. **Run**: shows the UI. Click Start and speak.

**Language mode.** Only the current language's commands are recognized. To switch, say a language's name in that language ("english", "español", "русский"). In the current language you can also name another one ("inglés" while in Spanish mode). You can also use the Language menu or type a name in the box next to it.

**Filters.** A command counts only if all of these pass:
- **Quiet before / quiet after**: the gaps around the command stay under the silence threshold, allowing for the gap tolerance. Set "quiet after" to 0 to skip it.
- **Min confidence**: Vosk's per-word confidence. With **decoy words** on, the grammar also contains a hundred or so common words, so the recognizer has real alternatives and its confidence stops sitting at 1.00.
- **Double-check**: the audio around the command is transcribed again by a less restricted engine: Vosk with its full vocabulary, or optionally Whisper. The transcript must *sound like* one of the command's phrases (**match similarity**). Near misses such as "твой год" for "твой ход" pass; unrelated words don't.

**Show vocabulary** lists exactly what can be said in the current mode.

Every candidate is logged with the reason it was accepted or rejected, and is marked on the level plot.

In [ ]:
import io
import json
import queue
import threading
import time
from math import gcd
from pathlib import Path

import numpy as np
import sounddevice as sd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from scipy.signal import butter, resample_poly, sosfilt, sosfilt_zi
from vosk import Model, KaldiRecognizer, SetLogLevel

SetLogLevel(-1)  # quiet Vosk logging (set to 0 to see grammar warnings)

VOSK_RATE = 16000     # rate the recognizers expect
BLOCK_SECONDS = 0.1   # audio chunk size handed to the recognizers
PLOT_SECONDS = 6.0    # length of the scrolling level / waveform display

MIC_DEVICE = None     # input device index (see the device list below); None = system default
MIC_GAIN = 1.0        # software gain; the mic tester below suggests a value
TEST_SECONDS = 5.0    # length of the mic test recording

# Starting values for the listener's tuning sliders.
DEFAULT_LANGUAGE = "en"
SILENCE_DB = -45.0       # a 20 ms frame quieter than this (dBFS, after gain) counts as silence
PRE_SILENCE_S = 1.0      # required quiet gap before a command
POST_SILENCE_S = 0.5     # required quiet gap after a command (0 = not required)
SILENCE_TOLERANCE = 10   # % of frames in a quiet gap allowed above the threshold (clicks, breaths)
MIN_CONFIDENCE = 0.60    # minimum per-word recognition confidence
USE_DECOYS = True        # add common non-command words to the grammar so confidence means something
VERIFY_ENGINE = "vosk"   # double-check engine: "off", "vosk" (full vocabulary) or "whisper"
MATCH_SIMILARITY = 0.75  # how close the double-check transcript must be to a command (0..1)
WHISPER_MODEL = "small"  # tiny / base / small / medium; bigger = more accurate, slower

# Command sets. Each maps a game command to the phrases that trigger it, per language.
# "your turn" is the player handing the turn to the computer.
# "original" is the first word list; "distinct" uses longer words chosen to sound unlike
# each other and unlike everyday speech. Switch between them live in the listener.
# Add variants freely; the model-load cell warns about words a model doesn't know.
COMMAND_SETS = {
    "original": {
        "en": {"start": ["start"], "done": ["done"], "quit": ["quit"],
               "reset": ["reset"], "your turn": ["your turn"]},
        "es": {"start": ["empezar", "iniciar"], "done": ["listo", "terminado"],
               "quit": ["salir"], "reset": ["reiniciar"],
               "your turn": ["tu turno", "es tu turno"]},
        "ru": {"start": ["начать", "старт"], "done": ["готово", "закончил"],
               "quit": ["выход", "выйти"], "reset": ["сброс", "сбросить"],
               "your turn": ["твоя очередь", "твой ход"]},
    },
    "distinct": {
        "en": {"start": ["begin"], "done": ["finished"], "quit": ["goodbye"],
               "reset": ["restart"], "your turn": ["your move"]},
        "es": {"start": ["comenzar", "empezar"], "done": ["terminado"],
               "quit": ["abandonar"], "reset": ["reiniciar"],
               "your turn": ["tu jugada"]},
        "ru": {"start": ["старт", "поехали"], "done": ["готово"],
               "quit": ["завершить"], "reset": ["перезапуск"],
               "your turn": ["твоя очередь"]},
    },
}
COMMAND_SET = "distinct"
commands = COMMAND_SETS[COMMAND_SET]
model_paths = {
    "en": "models/vosk-model-small-en-us-0.15",
    "es": "models/vosk-model-small-es-0.42",
    "ru": "models/vosk-model-small-ru-0.22",
}

LANG_LABELS = {"en": "English", "es": "Español", "ru": "Русский"}

# Spoken language names, per recognizer model.
# language_names[model_lang][target_lang] = phrases that switch to target_lang.
# The current language's model listens for all of its row (so "inglés" works in Spanish
# mode). The other models listen only for their own name ("español" said in Spanish).
language_names = {
    "en": {"en": ["english"], "es": ["spanish"], "ru": ["russian"]},
    "es": {"es": ["español"], "en": ["inglés"], "ru": ["ruso"]},
    "ru": {"ru": ["русский"], "en": ["английский"], "es": ["испанский"]},
}

# Decoy words: common words added to the grammar. Without them, the restricted grammar has
# to explain every sound as a command or [unk], so it reports ~1.00 confidence even when
# wrong. Words that sound too much like a command, or that a model doesn't know, are dropped
# automatically when the models load.
decoy_words = {
    "en": """the a and to of in is it that he was for on are with as his they be at one have
        this from or had by not but what some we can out other were all there when up use
        how said an each she which do their time if will way about many then them would
        like so these her long make thing see him two has look more day could go come did
        my no most number who over know water than call first people may down side been now
        find any new work part take get place made live where after back little only man
        year came show every good me give our under name very through just form much great
        think say help line before same right old too does tell set three want well also
        play small end put home read hand even here must big high such why ask men went
        light kind off need house try again point world near hello okay yes yeah well
        maybe sorry thanks please wait look listen game ready next stop open close""".split(),
    "es": """el la de que y a en un ser se no haber por con su para como estar tener le lo
        todo pero más hacer o poder decir este ir otro ese si me ya ver porque dar cuando
        él muy sin vez mucho saber qué sobre mi alguno mismo yo también hasta año dos
        querer entre así primero desde grande eso ni nos llegar pasar tiempo ella sí día
        uno bien poco deber entonces poner cosa tanto hombre parecer nuestro tan donde
        ahora parte después vida quedar siempre creer hablar llevar dejar nada cada seguir
        menos nuevo encontrar algo solo casa mundo país hora noche agua mano trabajo
        gracias hola bueno claro mira vamos oye perdón señor mujer niño juego mesa
        espera vale ahí aquí allí pues bueno nunca otra vez""".split(),
    "ru": """и в не на я быть он с что а по это она этот к но они мы как из у который то за
        весь от так о для ты же все тот мочь вы человек такой его сказать только или
        ещё бы себя один какой когда уже до время если сам нет да другой там потом очень
        вот где теперь день жизнь хотеть рука можно даже работа слово дело глаз раз здесь
        нужно спасибо привет хорошо пожалуйста дом друг вопрос сейчас ничего надо знать
        говорить думать видеть идти делать сидеть понять первый новый большой хороший
        последний ну ладно давай смотри слушай игра правильно неправильно может""".split(),
}

In [ ]:
import math
import unicodedata

def load_vocab(model_dir):
    """Return the model's word list if it ships one, else None."""
    for name in ("graph/words.txt", "words.txt"):
        p = Path(model_dir) / name
        if p.exists():
            with open(p, encoding="utf-8") as f:
                return {line.split()[0] for line in f if line.strip()}
    return None


# ---------- sound-alike comparison ----------
def _strip_accents(s):
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c))


_RU_FOLD = str.maketrans({
    # devoice consonant pairs, merge г/к/х, reduce vowels, drop soft/hard signs
    "б": "п", "в": "ф", "г": "к", "д": "т", "ж": "ш", "з": "с", "х": "к", "щ": "ш",
    "ц": "тс", "о": "а", "е": "и", "я": "и", "э": "и", "ы": "и", "й": "и", "ю": "у",
    "ь": None, "ъ": None})


def phon_fold(text, lang):
    """Rough 'how it sounds' spelling, so near-miss transcripts compare as close.
    ru: devoicing, г/к/х merged, unstressed-vowel style reduction, ь/ъ dropped
        ('твой год' == 'твой ход', 'вы год' == 'выход').
    es: silent h dropped, v->b, ll->y, z/ce/ci->s, qu/c->k.
    Spaces are dropped too, since word boundaries in transcripts are unreliable."""
    t = text.lower().replace("ё", "е")
    if lang == "es":
        t = _strip_accents(t).replace("ñ", "n")
        t = t.replace("ll", "y").replace("qu", "k").replace("ce", "se").replace("ci", "si")
        t = t.replace("c", "k").replace("z", "s").replace("v", "b").replace("h", "")
    elif lang == "ru":
        t = t.translate(_RU_FOLD)
    else:
        t = _strip_accents(t)
    return "".join(c for c in t if c.isalpha())


def similarity(a, b):
    """1 - normalized edit distance (1.0 = identical)."""
    if a == b:
        return 1.0
    if not a or not b:
        return 0.0
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return 1.0 - prev[-1] / max(len(a), len(b))


def best_window_match(heard_words, targets, lang):
    """Best similarity between any target phrase and any run of heard words of about
    the same length. Returns (score, matched_text, word_indices)."""
    best = (0.0, "", ())
    for tgt in targets:
        ft = phon_fold(tgt, lang)
        n = len(tgt.split())
        for k in sorted({max(1, n - 1), n, n + 1}):
            for i in range(max(1, len(heard_words) - k + 1)):
                chunk = heard_words[i:i + k]
                if not chunk:
                    continue
                sc = similarity(phon_fold(" ".join(chunk), lang), ft)
                if sc > best[0]:
                    best = (sc, " ".join(chunk), tuple(range(i, i + len(chunk))))
    return best


# ---------- load models, check vocabulary, prepare decoys ----------
def command_phrases(lang):
    """Every command phrase and language name the active-language model listens for."""
    out = [p for v in commands[lang].values() for p in v]
    out += [p for v in language_names[lang].values() for p in v]
    return [p.lower() for p in dict.fromkeys(out)]


models, vocabs, decoys = {}, {}, {}
for lang, path in model_paths.items():
    p = Path(path)
    if not p.is_dir():
        raise FileNotFoundError(f"{lang}: model folder not found at {p.resolve()}")
    models[lang] = Model(str(p))
    vocabs[lang] = load_vocab(p)
    if vocabs[lang] is None:
        print(f"{lang}: loaded (no word list to check against)")
        continue
    for set_name, cset in COMMAND_SETS.items():
        words = {w for v in cset[lang].values() for ph in v for w in ph.lower().split()}
        words |= {w for v in language_names[lang].values() for ph in v for w in ph.lower().split()}
        missing = sorted(w for w in words if w not in vocabs[lang])
        if missing:
            print(f"{lang} [{set_name}]: NOT in the model, will never be recognized: {missing}")
        else:
            print(f"{lang} [{set_name}]: all words are in the vocabulary")


def compute_decoys():
    """Decoys must be known to the model and must not sound like a current command word."""
    for lang in models:
        cmd_tokens = {w for ph in command_phrases(lang) for w in ph.split()}
        kept = []
        for w in dict.fromkeys(x.lower() for x in decoy_words.get(lang, [])):
            if vocabs[lang] is not None and w not in vocabs[lang]:
                continue
            if any(similarity(phon_fold(w, lang), phon_fold(c, lang)) >= 0.6 for c in cmd_tokens):
                continue
            kept.append(w)
        decoys[lang] = kept


def use_command_set(name):
    """Make COMMAND_SETS[name] the active commands (the listener rebuilds its grammars)."""
    global commands, COMMAND_SET
    COMMAND_SET, commands = name, COMMAND_SETS[name]
    compute_decoys()


use_command_set(COMMAND_SET)
print("decoy words available:", {l: len(v) for l, v in decoys.items()})


def grammar_phrases(lang, active, use_decoys):
    """Active language: its commands, language names and (optionally) decoys.
    Inactive languages: only their own name, so you can switch to them by voice."""
    if active:
        out = command_phrases(lang) + (decoys[lang] if use_decoys else [])
    else:
        out = [p.lower() for p in language_names[lang][lang]]
    return list(dict.fromkeys(out))


def make_recognizer(lang, active, use_decoys=USE_DECOYS):
    """Grammar-restricted recognizer with per-word times and confidences."""
    grammar = json.dumps(grammar_phrases(lang, active, use_decoys) + ["[unk]"], ensure_ascii=False)
    rec = KaldiRecognizer(models[lang], VOSK_RATE, grammar)
    rec.SetWords(True)
    return rec


def make_open_recognizer(lang):
    """Unrestricted recognizer (full vocabulary), used to double-check candidates."""
    rec = KaldiRecognizer(models[lang], VOSK_RATE)
    rec.SetWords(True)
    return rec


def phrase_table(lang, active):
    """[(tokens, kind, target)], longest phrases first.
    kind is 'command' (target = command name) or 'language' (target = language code)."""
    rows = []
    if active:
        for cmd, variants in commands[lang].items():
            rows += [(v.lower().split(), "command", cmd) for v in variants]
        for tgt, variants in language_names[lang].items():
            rows += [(v.lower().split(), "language", tgt) for v in variants]
    else:
        rows += [(v.lower().split(), "language", lang) for v in language_names[lang][lang]]
    return sorted(rows, key=lambda r: -len(r[0]))


def targets_for(lang, kind, target):
    """All phrases that mean the same thing (any variant of the command counts)."""
    if kind == "command":
        return [v.lower() for v in commands[lang][target]]
    return [v.lower() for v in language_names[lang][target]]


def find_phrases(words, table):
    """Non-overlapping phrase matches in a Vosk word list: [(i, j, kind, target)]."""
    toks = [w["word"].lower() for w in words]
    used = [False] * len(toks)
    out = []
    for ptoks, kind, tgt in table:
        n = len(ptoks)
        for i in range(len(toks) - n + 1):
            if toks[i:i + n] == ptoks and not any(used[i:i + n]):
                out.append((i, i + n, kind, tgt))
                used[i:i + n] = [True] * n
    return sorted(out)


def name_to_lang(text):
    """Map typed text ('es', 'Spanish', 'espanol', 'Русский', ...) to a language code."""
    fold = lambda s: _strip_accents(s.strip().lower())
    t = fold(text)
    if t in model_paths:
        return t
    for row in language_names.values():
        for tgt, names in row.items():
            if t in {fold(n) for n in names} or t == fold(LANG_LABELS[tgt]):
                return tgt
    return None


def clean_text(text):
    return " ".join(w for w in text.lower().split() if w != "[unk]")


# ---------- optional Whisper double-check ----------
try:
    import faster_whisper  # noqa: F401
    WHISPER_AVAILABLE = True
except ImportError:
    WHISPER_AVAILABLE = False
    if VERIFY_ENGINE == "whisper":
        print("faster-whisper is not installed; double-check falls back to Vosk. "
              "Install with:  pip install faster-whisper")
        VERIFY_ENGINE = "vosk"

_whisper = {"model": None, "loading": False, "error": None}

def load_whisper_async(on_done=None):
    """Load the Whisper model on a background thread (first use downloads it)."""
    if _whisper["model"] is not None or _whisper["loading"]:
        return
    _whisper["loading"] = True

    def work():
        try:
            from faster_whisper import WhisperModel
            _whisper["model"] = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
        except Exception as e:
            _whisper["error"] = repr(e)
        finally:
            _whisper["loading"] = False
            if on_done:
                on_done()
    threading.Thread(target=work, daemon=True).start()


def whisper_transcribe(pcm, lang):
    """Transcribe int16 16 kHz audio. Returns (words, mean token probability)."""
    audio = pcm.astype(np.float32) / 32768.0
    segments, _ = _whisper["model"].transcribe(
        audio, language=lang, beam_size=5, condition_on_previous_text=False,
        without_timestamps=True, vad_filter=False)
    segments = list(segments)
    text = " ".join(s.text for s in segments)
    words = "".join(c if (c.isalpha() or c.isspace()) else " " for c in text.lower()).split()
    prob = math.exp(sum(s.avg_logprob for s in segments) / len(segments)) if segments else 0.0
    return words, prob

In [ ]:
host_apis = sd.query_hostapis()

def list_inputs():
    return [
        (i, d, host_apis[d["hostapi"]]["name"])
        for i, d in enumerate(sd.query_devices())
        if d["max_input_channels"] > 0
    ]


def ac_stats(x):
    """Stats with the DC offset removed. A flat line (DC only) is not audio."""
    dc = float(np.mean(x))
    ac = x - dc
    return dc, float(np.sqrt(np.mean(ac ** 2))), float(np.max(np.abs(ac)))


def record(device, seconds):
    info = sd.query_devices(device, "input")
    rate = int(info["default_samplerate"])
    x = sd.rec(int(seconds * rate), samplerate=rate, channels=1,
               dtype="float32", device=device)
    sd.wait()
    return x[:, 0], rate, info


def suggest_gain(ac_peak, target=0.5, max_gain=100.0):
    """Gain that brings the loudest speech peak to about half of full scale."""
    if ac_peak < 1e-4:
        return None
    return float(np.clip(target / ac_peak, 1.0, max_gain))


def scan_inputs(seconds=1.5):
    """Record briefly from every input; keep talking the whole time."""
    print(f"Scanning every input for {seconds} s each. Keep talking (count out loud)...\n")
    print(f"{'idx':>3}  {'api':<20} {'speech RMS':>10} {'peak':>7} {'DC':>8}  verdict   name")
    for i, d, api in list_inputs():
        try:
            x, rate, _ = record(i, seconds)
            dc, rms, peak = ac_stats(x)
            verdict = "SIGNAL" if peak > 0.01 else ("weak" if peak > 1e-3 else "flat")
            print(f"{i:3d}  {api:<20} {rms:10.5f} {peak:7.4f} {dc:8.4f}  {verdict:<8}  {d['name']}")
        except Exception as e:
            print(f"{i:3d}  {api:<20} {'error':>10}  {str(e)[:60]}  {d['name']}")
    print("\nPick a device marked SIGNAL (largest peak while you talk) in the tester below.")


for i, d, api in list_inputs():
    print(f"{i:3d}  [{api}]  {d['name']}  (default {d['default_samplerate']:.0f} Hz)")
print("\nSystem default input device:", sd.default.device[0])

In [ ]:
# Talk continuously while this runs (about 1.5 s per device).
scan_inputs(seconds=1.5)

In [ ]:
# Interactive mic tester: choose device, duration and gain, then Record.
# "Use these settings" passes the device and gain to the listener cell.
_ids = [i for i, _, _ in list_inputs()]
tester_dev = widgets.Dropdown(
    options=[(f"{i}: [{api}] {d['name']}", i) for i, d, api in list_inputs()],
    value=MIC_DEVICE if MIC_DEVICE in _ids else _ids[0],
    description="Mic:", layout=widgets.Layout(width="760px"),
)
tester_secs = widgets.FloatSlider(value=TEST_SECONDS, min=1, max=15, step=0.5,
                                  description="Seconds:", readout_format=".1f")
tester_gain = widgets.FloatLogSlider(value=max(MIC_GAIN, 1.0), base=10, min=0, max=2,
                                     step=0.02, description="Gain (x):", readout_format=".1f",
                                     layout=widgets.Layout(width="500px"))
tester_rec = widgets.Button(description="Record", button_style="primary", icon="microphone")
tester_suggest = widgets.Button(description="Apply suggested gain")
tester_use = widgets.Button(description="Use these settings", button_style="success")
tester_msg = widgets.HTML("Pick a mic, click Record, and talk for the whole duration.")
tester_out = widgets.Output()
_last = {"suggested": None, "x": None, "rate": None, "name": ""}


def _plot():
    x, rate = _last["x"], _last["rate"]
    if x is None:
        return
    dc, rms, peak = ac_stats(x)
    ac = x - dc
    g = tester_gain.value
    y = np.clip(ac * g, -1, 1)
    clipped = float(np.mean(np.abs(ac * g) >= 1.0)) * 100
    t = np.arange(len(x)) / rate
    with tester_out:
        tester_out.clear_output(wait=True)
        print(f"Raw:  DC offset {dc:+.4f}   speech RMS {rms:.5f}   peak {peak:.4f}")
        if _last["suggested"] is None:
            print("Flat line: this device is not delivering audio. Try another (run scan_inputs()).")
        else:
            print(f"Suggested gain {_last['suggested']:.1f}x.  At {g:.1f}x: "
                  f"peak {float(np.max(np.abs(y))):.3f}, {clipped:.2f}% clipped")
        fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
        axes[0].plot(t, x, lw=0.5)
        axes[0].set_title(f"Raw (with DC offset): {_last['name']}")
        axes[0].set_ylim(min(-0.05, x.min() * 1.2), max(0.05, x.max() * 1.2))
        axes[1].plot(t, y, lw=0.5, color="tab:green")
        axes[1].set_title(f"DC removed, {g:.1f}x gain")
        axes[1].set_ylim(-1, 1)
        axes[1].set_xlabel("seconds")
        fig.tight_layout()
        plt.show()


def _on_record(_):
    tester_rec.disabled = True
    tester_msg.value = f"<b>Recording {tester_secs.value:.1f} s... talk now</b>"
    try:
        x, rate, info = record(tester_dev.value, tester_secs.value)
        _last.update(x=x, rate=rate, name=info["name"],
                     suggested=suggest_gain(ac_stats(x)[2]))
        tester_msg.value = "Done. Move the gain slider to preview; it redraws the plot."
        _plot()
    except Exception as e:
        tester_msg.value = f"<b style='color:#c00'>Could not record: {e}</b>"
    finally:
        tester_rec.disabled = False


def _on_suggest(_):
    if _last["suggested"]:
        tester_gain.value = _last["suggested"]


def _on_use(_):
    global MIC_DEVICE, MIC_GAIN
    MIC_DEVICE, MIC_GAIN = tester_dev.value, round(tester_gain.value, 1)
    tester_msg.value = (f"Saved: MIC_DEVICE = {MIC_DEVICE}, MIC_GAIN = {MIC_GAIN}x. "
                        "Now run the listener cells.")


tester_rec.on_click(_on_record)
tester_suggest.on_click(_on_suggest)
tester_use.on_click(_on_use)
tester_gain.observe(lambda ch: _plot(), names="value")

display(widgets.VBox([
    tester_dev,
    widgets.HBox([tester_secs, tester_gain]),
    widgets.HBox([tester_rec, tester_suggest, tester_use]),
    tester_msg, tester_out,
]))

In [ ]:
FRAME = 320                 # 20 ms level frames at 16 kHz
FPS = VOSK_RATE // FRAME    # 50 frames per second
GUARD_S = 0.15              # skip this much next to the word edges (word timings are approximate)
NOISE_MARGIN_DB = 8.0       # "Set from noise floor" puts the threshold this far above the floor
PCM_RING_S = 15             # seconds of audio kept for the open-vocabulary check
LEVEL_RING_S = 30           # seconds of level history kept


class CommandListener:
    """Background voice-command listener with a language mode, silence gates and a
    confidence threshold. Nothing here blocks the notebook."""

    def __init__(self, on_command=None, on_language=None, debounce_s=1.0):
        self.on_command = on_command      # fn(cmd, lang, phrase)
        self.on_language = on_language    # fn(lang)
        self.debounce_s = debounce_s
        self.lang = DEFAULT_LANGUAGE
        self._req_lang = None
        self._req_rebuild = False
        self._syncing = False
        self.stream = None
        self.worker = None
        self.stop_event = threading.Event()
        self.q = queue.Queue(maxsize=50)
        self.log_lines = []
        self.noise_floor = None
        self._reset_buffers()
        self._build_ui()

    # ================= buffers =================
    def _reset_buffers(self):
        self.total = 0                                    # 16 kHz samples processed
        self.pcm_ring = np.zeros(PCM_RING_S * VOSK_RATE, np.int16)
        self.db_ring = np.full(LEVEL_RING_S * FPS, -120.0)
        self.pk_ring = np.zeros(LEVEL_RING_S * FPS)
        self.nframes = 0
        self._fbuf = np.zeros(0, np.float32)
        self.candidates = []                              # waiting for post-silence audio
        self.marks = []                                   # [start, end, color] for the plot
        self._last_clip = 0.0
        self._time_mode = None                            # see _time_base
        self._last_cmd = (None, -10 ** 12)

    def _push(self, pcm):
        N = len(self.pcm_ring)
        self.pcm_ring[(self.total + np.arange(len(pcm))) % N] = pcm
        self.total += len(pcm)
        f = np.concatenate([self._fbuf, pcm.astype(np.float32) / 32768.0])
        k = len(f) // FRAME
        if k:
            fr = f[:k * FRAME].reshape(k, FRAME)
            idx = (self.nframes + np.arange(k)) % len(self.db_ring)
            self.db_ring[idx] = 20 * np.log10(np.sqrt(np.mean(fr ** 2, axis=1)) + 1e-6)
            self.pk_ring[idx] = np.abs(fr).max(axis=1)
            self.nframes += k
        self._fbuf = f[k * FRAME:]

    def _get_pcm(self, a, b):
        N = len(self.pcm_ring)
        a, b = max(a, self.total - N, 0), min(b, self.total)
        if b <= a:
            return np.zeros(0, np.int16)
        return self.pcm_ring[np.arange(a, b) % N]

    def _recent_frames(self, seconds):
        n = min(int(seconds * FPS), self.nframes, len(self.db_ring))
        idx = np.arange(self.nframes - n, self.nframes) % len(self.db_ring)
        return self.db_ring[idx], self.pk_ring[idx]

    def _loud_fraction(self, a, b):
        """Fraction of 20 ms frames in samples [a, b) above the silence threshold.
        Time before the stream started counts as silent."""
        fa = max(a // FRAME, 0, self.nframes - len(self.db_ring))
        fb = min(b // FRAME, self.nframes)
        if fb <= fa:
            return 0.0
        db = self.db_ring[np.arange(fa, fb) % len(self.db_ring)]
        return float(np.mean(db > self.sil_slider.value))

    # ================= UI =================
    def _build_ui(self):
        inputs = list_inputs()
        if not inputs:
            raise RuntimeError("No microphone input devices found.")
        ids = [i for i, _, _ in inputs]
        preferred = MIC_DEVICE if MIC_DEVICE is not None else sd.default.device[0]
        W = widgets.Layout
        self.device_dd = widgets.Dropdown(
            options=[(f"{i}: [{api}] {d['name']}", i) for i, d, api in inputs],
            value=preferred if preferred in ids else ids[0],
            description="Mic:", layout=W(width="760px"))
        self.start_btn = widgets.Button(description="Start", button_style="success")
        self.stop_btn = widgets.Button(description="Stop", button_style="danger", disabled=True)
        self.start_btn.on_click(self.start)
        self.stop_btn.on_click(self.stop)
        self.level = widgets.FloatProgress(min=0, max=1, description="Level:", layout=W(width="400px"))
        self.gain_slider = widgets.FloatLogSlider(
            value=min(max(MIC_GAIN, 1.0), 100.0), base=10, min=0, max=2, step=0.02,
            description="Gain (x):", readout_format=".1f", layout=W(width="460px"))
        self.clip_label = widgets.HTML("")

        # Language mode
        self.lang_dd = widgets.Dropdown(options=[(LANG_LABELS[l], l) for l in models],
                                        value=self.lang, description="Language:")
        self.lang_dd.observe(self._on_lang_dd, names="value")
        self.lang_text = widgets.Text(placeholder="or type a name, e.g. español / russian",
                                      continuous_update=False, layout=W(width="300px"))
        self.lang_text.observe(self._on_lang_text, names="value")
        self.mode_label = widgets.HTML("")
        self._update_mode_label()

        # Gates
        st = {"description_width": "150px"}
        wide = W(width="520px")
        self.sil_slider = widgets.FloatSlider(value=SILENCE_DB, min=-90, max=-10, step=1,
            description="Silence threshold dBFS", style=st, layout=wide, readout_format=".0f")
        self.pre_slider = widgets.FloatSlider(value=PRE_SILENCE_S, min=0, max=3, step=0.1,
            description="Quiet before (s)", style=st, layout=wide, readout_format=".1f")
        self.post_slider = widgets.FloatSlider(value=POST_SILENCE_S, min=0, max=3, step=0.1,
            description="Quiet after (s, 0=off)", style=st, layout=wide, readout_format=".1f")
        self.tol_slider = widgets.IntSlider(value=SILENCE_TOLERANCE, min=0, max=40, step=1,
            description="Gap tolerance %", style=st, layout=wide)
        self.conf_slider = widgets.FloatSlider(value=MIN_CONFIDENCE, min=0, max=1, step=0.01,
            description="Min confidence", style=st, layout=wide, readout_format=".2f")
        self.cmdset_dd = widgets.Dropdown(options=list(COMMAND_SETS), value=COMMAND_SET,
                                          description="Command set:")
        self.cmdset_dd.observe(self._on_cmdset, names="value")
        self.decoy_cb = widgets.Checkbox(value=USE_DECOYS, indent=False,
            description="Decoy words in grammar (makes confidence meaningful)")
        self.decoy_cb.observe(self._on_decoy, names="value")
        engines = [("off", "off"), ("Vosk full vocabulary", "vosk")]
        engines.append(("Whisper", "whisper") if WHISPER_AVAILABLE
                       else ("Whisper (pip install faster-whisper)", "whisper-missing"))
        self.verify_dd = widgets.Dropdown(options=engines, description="Double-check",
            value=VERIFY_ENGINE if VERIFY_ENGINE in ("off", "vosk", "whisper") else "vosk",
            style=st, layout=W(width="420px"))
        self.verify_dd.observe(self._on_verify_engine, names="value")
        self.verify_note = widgets.HTML("")
        self.sim_slider = widgets.FloatSlider(value=MATCH_SIMILARITY, min=0.3, max=1, step=0.01,
            description="Match similarity", style=st, layout=wide, readout_format=".2f")

        # Vocabulary panel
        self.vocab_toggle = widgets.ToggleButton(value=False, description="Show vocabulary",
                                                 icon="list")
        self.vocab_toggle.observe(lambda ch: self._refresh_vocab(), names="value")
        self.vocab_html = widgets.HTML("")
        self.floor_btn = widgets.Button(description="Set threshold from noise floor",
                                        layout=W(width="240px"))
        self.floor_btn.on_click(self._set_from_floor)
        self.floor_label = widgets.HTML("noise floor: (start listening)")

        self.status = widgets.HTML("Choose a microphone, then click Start.")
        self.partial = widgets.HTML("")
        self.plot_img = widgets.Image(format="png")
        self.log = widgets.HTML("")
        self._build_figure()
        self._render()
        if self.verify_dd.value == "whisper":
            self._on_verify_engine({"new": "whisper"})

    def _build_figure(self):
        self.fig = Figure(figsize=(9, 3.2), dpi=100)
        FigureCanvasAgg(self.fig)
        self.ax_w, self.ax_db = self.fig.subplots(2, 1, sharex=True,
                                                  gridspec_kw={"height_ratios": [1, 1.4]})
        n = int(PLOT_SECONDS * FPS)
        self.plot_x = (np.arange(n) - n) / FPS
        (self.l_pk_hi,) = self.ax_w.plot(self.plot_x, np.zeros(n), lw=0.6, color="tab:blue")
        (self.l_pk_lo,) = self.ax_w.plot(self.plot_x, np.zeros(n), lw=0.6, color="tab:blue")
        self.ax_w.set_ylim(-1, 1)
        self.ax_w.set_ylabel("wave")
        (self.l_db,) = self.ax_db.plot(self.plot_x, np.full(n, -120.0), lw=0.8, color="tab:blue")
        self.l_thr = self.ax_db.axhline(SILENCE_DB, color="tab:red", lw=1, ls="--")
        self.ax_db.set_ylim(-90, 0)
        self.ax_db.set_xlim(self.plot_x[0], 0)
        self.ax_db.set_ylabel("dBFS")
        self.ax_db.set_xlabel("seconds ago   (orange = pending, green = accepted, red = rejected)")
        self._spans = []
        self.fig.tight_layout()

    def display(self):
        display(widgets.VBox([
            self.device_dd,
            widgets.HBox([self.start_btn, self.stop_btn, self.level]),
            widgets.HBox([self.gain_slider, self.clip_label]),
            widgets.HBox([self.lang_dd, self.lang_text, self.mode_label]),
            self.cmdset_dd,
            self.sil_slider,
            widgets.HBox([self.floor_btn, self.floor_label]),
            self.pre_slider, self.post_slider, self.tol_slider,
            self.conf_slider, self.decoy_cb,
            widgets.HBox([self.verify_dd, self.verify_note]), self.sim_slider,
            self.vocab_toggle, self.vocab_html,
            self.status, self.plot_img, self.partial, self.log,
        ]))

    def _render(self):
        db, pk = self._recent_frames(PLOT_SECONDS)
        n = len(self.plot_x)
        full_db = np.full(n, -120.0)
        full_pk = np.zeros(n)
        if len(db):
            full_db[-len(db):] = db
            full_pk[-len(pk):] = pk
        self.l_db.set_ydata(full_db)
        self.l_pk_hi.set_ydata(full_pk)
        self.l_pk_lo.set_ydata(-full_pk)
        self.l_thr.set_ydata([self.sil_slider.value] * 2)
        for s in self._spans:
            s.remove()
        self._spans = []
        horizon = self.total - int(PLOT_SECONDS * VOSK_RATE)
        self.marks = [m for m in self.marks if m[1] > horizon]
        for a, b, color in self.marks:
            x0, x1 = (a - self.total) / VOSK_RATE, (b - self.total) / VOSK_RATE
            for ax in (self.ax_w, self.ax_db):
                self._spans.append(ax.axvspan(x0, x1, color=color, alpha=0.35))
        buf = io.BytesIO()
        self.fig.savefig(buf, format="png")
        self.plot_img.value = buf.getvalue()

    def _log(self, msg):
        self.log_lines = (self.log_lines + [f"{time.strftime('%H:%M:%S')}  {msg}"])[-20:]
        self.log.value = "<pre style='margin:0'>" + "\n".join(reversed(self.log_lines)) + "</pre>"

    def _update_mode_label(self):
        self.mode_label.value = (f"<b style='font-size:1.2em'>&nbsp;Mode: {LANG_LABELS[self.lang]}</b>"
                                 f" &nbsp;(say \"{language_names[self.lang][self.lang][0]}\" etc.)")

    def _set_from_floor(self, _=None):
        if self.noise_floor is not None:
            self.sil_slider.value = round(self.noise_floor + NOISE_MARGIN_DB)

    # ================= language mode =================
    def set_language(self, lang):
        """Switch mode from code or the UI. Safe to call while listening."""
        if lang not in models:
            raise ValueError(f"unknown language {lang!r}")
        if self.worker is not None:
            self._req_lang = lang            # the worker thread applies it
        else:
            self.lang = lang
            self._sync_lang_widgets()

    def _on_lang_dd(self, change):
        if not self._syncing:
            self.set_language(change["new"])

    def _on_lang_text(self, change):
        text = change["new"].strip()
        if not text:
            return
        lang = name_to_lang(text)
        if lang is None:
            self.status.value = f"<b style='color:#c60'>Unknown language: {text!r}</b>"
        else:
            self.set_language(lang)
        self.lang_text.value = ""

    def _sync_lang_widgets(self):
        self._syncing = True
        self.lang_dd.value = self.lang
        self._syncing = False
        self._update_mode_label()
        self._refresh_vocab()

    def _refresh_vocab(self):
        if not self.vocab_toggle.value:
            self.vocab_html.value = ""
            return
        L = self.lang
        sep = " &nbsp;|&nbsp; "
        rows = "".join(f"<tr><td><b>{cmd}</b>&nbsp;&nbsp;</td><td>{sep.join(v)}</td></tr>"
                       for cmd, v in commands[L].items())
        here = sep.join(f"{p} &rarr; {LANG_LABELS[t]}"
                        for t, v in language_names[L].items() for p in v)
        other = sep.join(f"{p} &rarr; {LANG_LABELS[o]}"
                         for o in models if o != L for p in language_names[o][o])
        n_dec = len(decoys[L]) if self.decoy_cb.value else 0
        dec = ", ".join(decoys[L]) if self.decoy_cb.value else "(off)"
        self.vocab_html.value = (
            "<div style='border:1px solid #ccc;padding:6px;max-width:900px'>"
            f"<b>Commands ({LANG_LABELS[L]} mode)</b><table>{rows}</table>"
            f"<b>Switch language, said in {LANG_LABELS[L]}:</b> {here}<br>"
            f"<b>Switch language, said in that language:</b> {other}<br>"
            f"<details><summary>Decoy words ({n_dec}): recognized but ignored</summary>"
            f"<small>{dec}</small></details></div>")

    def _on_cmdset(self, change):
        use_command_set(change["new"])
        if self.worker is not None:
            self._req_rebuild = True
        self._update_mode_label()
        self._refresh_vocab()
        self._log(f"command set '{change['new']}'")

    def _on_decoy(self, change):
        self._refresh_vocab()
        if self.worker is not None:
            self._req_rebuild = True
        self._log(f"decoy words {'on' if change['new'] else 'off'}")

    def _on_verify_engine(self, change):
        eng = change["new"]
        if eng == "whisper-missing":
            self.verify_note.value = ("<span style='color:#c60'>run  pip install faster-whisper"
                                      "  and restart the kernel</span>")
        elif eng == "whisper" and _whisper["model"] is None:
            self.verify_note.value = "loading Whisper (the first time downloads the model)..."
            load_whisper_async(on_done=self._whisper_loaded)
        else:
            self.verify_note.value = ""

    def _whisper_loaded(self):
        if _whisper["error"]:
            self.verify_note.value = f"<span style='color:#c00'>Whisper failed: {_whisper['error']}</span>"
        else:
            self.verify_note.value = f"Whisper '{WHISPER_MODEL}' ready"

    def _build_recognizers(self):
        self.recs = {}
        for lang in models:
            active = lang == self.lang
            self.recs[lang] = {"rec": make_recognizer(lang, active, self.decoy_cb.value),
                               "origin": self.total, "utt_start": self.total}
        self.tables = {lang: phrase_table(lang, lang == self.lang) for lang in models}

    def _switch(self, lang, how):
        old = self.lang
        self.lang = lang
        self._build_recognizers()
        self.candidates = []
        self._dropped_pending = True
        self._sync_lang_widgets()
        self._log(f"LANGUAGE {LANG_LABELS[old]} -> {LANG_LABELS[lang]} ({how})")
        if self.on_language:
            self.on_language(lang)

    # ================= audio =================
    def _callback(self, indata, frames, time_info, status):
        if status:
            self.last_status = str(status)
        try:
            self.q.put_nowait(indata[:, 0].copy())
        except queue.Full:
            self.dropped += 1

    def _pick_rate(self, dev):
        try:
            sd.check_input_settings(device=dev, samplerate=VOSK_RATE, channels=1, dtype="float32")
            return VOSK_RATE
        except Exception:
            return int(sd.query_devices(dev)["default_samplerate"])

    def start(self, _=None):
        if self.stream is not None:
            return
        dev = self.device_dd.value
        self.in_rate = self._pick_rate(dev)
        self.q = queue.Queue(maxsize=50)
        self.blocks = self.dropped = 0
        self.last_status = ""
        self._reset_buffers()
        self._prepare_dsp()
        self._build_recognizers()
        self.stop_event.clear()
        try:
            self.stream = sd.InputStream(
                device=dev, samplerate=self.in_rate, channels=1, dtype="float32",
                blocksize=int(self.in_rate * BLOCK_SECONDS), callback=self._callback)
            self.stream.start()
        except Exception as e:
            self.stream = None
            self.status.value = f"<b style='color:#c00'>Could not open device {dev}: {e}</b>"
            return
        self.worker = threading.Thread(target=self._worker, daemon=True)
        self.worker.start()
        self.start_btn.disabled, self.stop_btn.disabled = True, False
        self.device_dd.disabled = True
        self._log(f"Started device {dev} at {self.in_rate} Hz, mode {LANG_LABELS[self.lang]}")

    def stop(self, _=None):
        self.stop_event.set()
        if self.stream is not None:
            self.stream.stop()
            self.stream.close()
            self.stream = None
        if self.worker is not None:
            self.worker.join(timeout=3)
            self.worker = None
        if self._req_lang:                     # menu change made while stopping
            self.lang, self._req_lang = self._req_lang, None
            self._sync_lang_widgets()
        self.start_btn.disabled, self.stop_btn.disabled = False, True
        self.device_dd.disabled = False
        self.level.value = 0
        self.clip_label.value = ""
        self.status.value = "Stopped."

    def _prepare_dsp(self):
        g = gcd(VOSK_RATE, self.in_rate)
        self._up, self._down = VOSK_RATE // g, self.in_rate // g
        # 60 Hz high-pass removes DC offset and rumble before the gain is applied.
        self._hp = butter(2, 60, btype="highpass", fs=self.in_rate, output="sos")
        self._hp_state = None
        self._last_draw = 0.0
        self.partials = {}

    def _worker(self):
        try:
            while not self.stop_event.is_set():
                try:
                    block = self.q.get(timeout=0.2)
                except queue.Empty:
                    continue
                self.process_block(block)
        except Exception as e:
            import traceback
            traceback.print_exc()
            self.status.value = f"<b style='color:#c00'>Worker error: {e!r}</b>"

    def process_block(self, block):
        """One audio block at self.in_rate (float32, mono). Called by the worker thread."""
        if self._req_lang:
            lang, self._req_lang = self._req_lang, None
            if lang != self.lang:
                self._switch(lang, "menu")
        if self._req_rebuild:
            self._req_rebuild = False
            self._build_recognizers()
            self.candidates = []

        if self._hp_state is None:
            self._hp_state = sosfilt_zi(self._hp) * block[0]
        block, self._hp_state = sosfilt(self._hp, block, zi=self._hp_state)
        if self.in_rate != VOSK_RATE:
            block = resample_poly(block, self._up, self._down)
        block = block * self.gain_slider.value
        if np.mean(np.abs(block) >= 1.0) > 0.001:
            self._last_clip = time.monotonic()
        pcm = np.clip(block * 32767, -32768, 32767).astype(np.int16)
        self.blocks += 1
        self._push(pcm)

        data = pcm.tobytes()
        for lang, info in list(self.recs.items()):
            rec = info["rec"]
            if rec.AcceptWaveform(data):
                self.partials[lang] = ""
                self._on_final(lang, info, rec.Result())
            elif lang == self.lang:
                self.partials[lang] = clean_text(json.loads(rec.PartialResult()).get("partial", ""))
        self._process_candidates()

        now = time.monotonic()
        if now - self._last_draw > 0.2:
            self._last_draw = now
            self._update_ui()

    def _update_ui(self):
        db, _ = self._recent_frames(0.2)
        cur = float(np.max(db)) if len(db) else -120.0
        hist, _ = self._recent_frames(10)
        if len(hist) > FPS:
            self.noise_floor = float(np.percentile(hist, 10))
            self.floor_label.value = (f"noise floor about {self.noise_floor:.0f} dBFS "
                                      f"(button sets threshold to floor + {NOISE_MARGIN_DB:.0f})")
        self.level.value = float(np.clip((cur + 90) / 90, 0, 1))
        self.clip_label.value = ("<b style='color:#c00'>&nbsp;CLIPPING: lower gain</b>"
                                 if time.monotonic() - self._last_clip < 0.5 else "")
        quiet = "quiet" if cur <= self.sil_slider.value else "<b>above threshold</b>"
        extra = f" &nbsp;<span style='color:#c60'>{self.last_status}</span>" if self.last_status else ""
        self.status.value = (f"Listening ({LANG_LABELS[self.lang]}) &middot; level {cur:.0f} dBFS ({quiet})"
                             f" &middot; pending {len(self.candidates)} &middot; dropped {self.dropped}{extra}")
        heard = self.partials.get(self.lang, "")
        self.partial.value = f"<i>hearing: {heard}</i>" if heard else ""
        self._render()

    # ================= recognition results =================
    def _time_base(self, info, utt_start, t_end):
        """Sample index that Vosk word times count from. Vosk versions differ: times are
        from recognizer creation or from the start of the utterance. The right base puts
        the utterance's last word end shortly before 'now'; once that's unambiguous it's cached."""
        R = VOSK_RATE
        bases = {"stream": info["origin"], "utterance": utt_start}
        if self._time_mode is None:
            def err(b):
                e = b + t_end * R
                return self.total - e if e <= self.total + 0.3 * R else float("inf")
            errs = {k: err(b) for k, b in bases.items()}
            mode = min(errs, key=errs.get)
            other = "utterance" if mode == "stream" else "stream"
            if utt_start - info["origin"] > 3 * R and errs[other] > 5 * R:
                self._time_mode = mode
                self._log(f"(word times count from {mode} start)")
            return bases[mode]
        return bases[self._time_mode]

    def _on_final(self, lang, info, result_json):
        res = json.loads(result_json)
        utt_start, info["utt_start"] = info["utt_start"], self.total
        words = [w for w in res.get("result", []) if "start" in w]
        if not words:
            return
        base = self._time_base(info, utt_start, float(words[-1]["end"]))
        matches = find_phrases(words, self.tables[lang])
        for i, j, kind, tgt in matches:
            mark = [base + int(words[i]["start"] * VOSK_RATE),
                    base + int(words[j - 1]["end"] * VOSK_RATE), "orange"]
            self.marks.append(mark)
            phrase = " ".join(w["word"] for w in words[i:j])
            self.candidates.append(dict(
                lang=lang, kind=kind, target=tgt, phrase=phrase, tokens=phrase.lower().split(),
                start=mark[0], end=mark[1], mark=mark, created=self.total,
                conf=min(float(w.get("conf", 1.0)) for w in words[i:j])))
        if not matches and lang == self.lang:
            text = clean_text(res.get("text", ""))
            if text:
                self._log(f"heard ({lang}): {text}")

    def _process_candidates(self):
        self._dropped_pending = False
        keep = []
        for c in list(self.candidates):
            if self._dropped_pending:
                break
            post = self.post_slider.value
            need = c["end"] + (int((GUARD_S + post) * VOSK_RATE) if post > 0 else 0)
            if self.total < need and self.total - c["created"] < 10 * VOSK_RATE:
                keep.append(c)
            else:
                self._decide(c)
        if not self._dropped_pending:
            self.candidates = keep

    def _gate(self, c):
        R, g = VOSK_RATE, int(GUARD_S * VOSK_RATE)
        tol = self.tol_slider.value / 100
        reasons, notes = [], []
        pre = self.pre_slider.value
        if pre > 0:
            f = self._loud_fraction(c["start"] - g - int(pre * R), c["start"] - g)
            notes.append(f"before {f:.0%} loud")
            if f > tol:
                reasons.append(f"not quiet before ({f:.0%} of {pre:.1f}s above threshold)")
        post = self.post_slider.value
        if post > 0:
            f = self._loud_fraction(c["end"] + g, c["end"] + g + int(post * R))
            notes.append(f"after {f:.0%} loud")
            if f > tol:
                reasons.append(f"not quiet after ({f:.0%} of {post:.1f}s above threshold)")
        thr = self.conf_slider.value
        if c["conf"] < thr:
            reasons.append(f"confidence {c['conf']:.2f} < {thr:.2f}")
        return reasons, notes

    def _verify(self, c):
        """Re-transcribe the audio around the candidate with a less restricted engine and
        compare it with the command's phrases by sound-alike similarity, so near misses
        ('начат', 'твой год') pass and unrelated speech doesn't. Returns (reasons, note)."""
        engine = self.verify_dd.value
        if engine in ("off", "whisper-missing"):
            return [], ""
        if engine == "whisper" and _whisper["model"] is None:
            engine = "vosk"                                   # still loading, or failed
        pad = int(0.4 * VOSK_RATE)
        seg = self._get_pcm(c["start"] - pad, c["end"] + pad)
        targets = targets_for(c["lang"], c["kind"], c["target"])
        confs, prob = None, None
        if engine == "whisper":
            heard, prob = whisper_transcribe(seg, c["lang"])
        else:
            rec = make_open_recognizer(c["lang"])
            rec.AcceptWaveform(seg.tobytes())
            res = json.loads(rec.FinalResult())
            words = [w for w in res.get("result", []) if w.get("word") != "[unk]"]
            heard = [w["word"].lower() for w in words]
            confs = [float(w.get("conf", 1.0)) for w in words]
        score, matched, idx = best_window_match(heard, targets, c["lang"])
        tag = "whisper" if engine == "whisper" else "vosk-full"
        heard_s = " ".join(heard)
        note = f"{tag} heard '{heard_s}' sim {score:.2f}"
        reasons = []
        if score < self.sim_slider.value:
            reasons.append(f"similarity {score:.2f} < {self.sim_slider.value:.2f}")
        elif confs is not None and idx:
            oconf = min(confs[i] for i in idx)
            note += f" conf {oconf:.2f}"
            if oconf < self.conf_slider.value:
                reasons.append(f"{tag} confidence {oconf:.2f} < {self.conf_slider.value:.2f}")
        elif prob is not None:
            note += f" p {prob:.2f}"
        return reasons, note

    def _decide(self, c):
        reasons, notes = self._gate(c)
        if not reasons:
            vreasons, vnote = self._verify(c)
            reasons += vreasons
            if vnote:
                notes.append(vnote)
        desc = f"'{c['phrase']}' ({c['lang']}, conf {c['conf']:.2f}, {', '.join(notes) or 'no gap check'})"
        c["mark"][2] = "tab:red" if reasons else "tab:green"
        if reasons:
            self._log(f"reject {desc}: {'; '.join(reasons)}")
            return
        if c["kind"] == "language":
            if c["target"] != self.lang:
                self._switch(c["target"], f"voice {desc}")
            return
        cmd = c["target"]
        if cmd == self._last_cmd[0] and c["start"] - self._last_cmd[1] < self.debounce_s * VOSK_RATE:
            self._log(f"(duplicate {cmd!r} within {self.debounce_s:.1f}s ignored)")
            return
        self._last_cmd = (cmd, c["end"])
        self._log(f"COMMAND {cmd!r}  {desc}")
        if self.on_command:
            self.on_command(cmd, c["lang"], c["phrase"])

### Tuning tips

1. Start listening and stay quiet for a few seconds. Click **Set threshold from noise floor**. The red dashed line on the dB plot should sit just above the flat quiet trace.
2. Talk normally, as if to another person. Your speech should rise well above the line. If the room noise crosses the line often, raise the threshold or the gap tolerance a little.
3. Say commands with a pause before them. Each one shows up as an orange span, then turns green (accepted) or red (rejected). The log says why.
4. If you get false positives from background conversation, raise **Quiet before**, turn on **Quiet after**, and raise **Min confidence**. If real commands are rejected, read the log reason and loosen only that setting.
5. The gain changes the dB levels. Set the gain first, then the threshold.

#### Accuracy

- **Confidence stuck at 1.00?** That happens when the grammar contains only the commands: the recognizer has nothing else to choose from. Turn on **decoy words** and confidence becomes informative. Raise **Min confidence** once you've seen typical values.
- **True commands rejected by the double-check?** The log shows what it heard and the similarity score. If it heard a near miss ("чат" for "начать"), lower **Match similarity** a little, around 0.65–0.7. If a phrase keeps decoding as something unrelated ("в этой" for "выйти"), the model handles that word badly: add a variant or pick a different word (see "Show vocabulary").
- **Whisper** is much more accurate than the small Vosk models, especially for Russian, but takes roughly 0.3–1.5 s per command on a CPU. Install it with `pip install faster-whisper`, restart the kernel, and pick it under **Double-check**. The first use downloads the model (`small` is about 0.5 GB).
- Short words, and words that are mostly unstressed vowels, are the hardest. Two-syllable words with distinct consonants (e.g. "старт", "готово") are the easiest to recognize.

In [ ]:
# Stop any previous listener before re-running this cell.
try:
    listener.stop()
except NameError:
    pass

listener = CommandListener()
listener.display()

# Hooks for the game, e.g.:
# listener.on_command  = lambda cmd, lang, phrase: game.handle(cmd)
# listener.on_language = lambda lang: game.set_language(lang)
# listener.set_language("es")   # switch from code